# Unidade 2 · Meu primeiro modelo, provado com honestidade
**Aprendizagem Computacional · 4º TIA · Apostila do Aluno · versão INICIAL**

Este notebook acompanha a **Unidade 2** da apostila (cap. 5 a 10). A ordem é a da apostila: **primeiro a régua, depois a preparação, depois os modelos, e por último a teoria que explica o que vimos acontecer**.

| Parte | Dados | Capítulos |
|---|---|---|
| A · A régua: separar e baseline | pinguins (da Unidade 1) | 5 |
| B · Preparar sem trapacear e a escala | pinguins | 6 e 7 |
| C · Regressão | diabetes (`load_diabetes`, 442 pacientes) | 8 |
| D · Classificação: logística, métricas e limiar | câncer de mama (`load_breast_cancer`, 569 casos) | 9 |
| E · Por que o treino engana | câncer de mama | 10 |

Regras: rode de cima para baixo; `random_state=42` em tudo que sorteia.

**Como usar a versão INICIAL:** procure os comentários `# TODO`. Complete o código e rode a célula. As células seguintes dependem das anteriores.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEMENTE = 42
from pathlib import Path
CSV_LOCAL = Path('../dados/penguins.csv')   # cópia no repositório: funciona sem internet
if CSV_LOCAL.exists():
    pinguins = pd.read_csv(CSV_LOCAL)
else:
    pinguins = sns.load_dataset('penguins')   # Colab: baixa da internet
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings('ignore', category=UndefinedMetricWarning)  # o baseline nunca prevê 'maligno'
pd.set_option('display.precision', 3)

ALVO = 'species'
NUMERICAS = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
CATEGORICAS = ['island', 'sex']
X = pinguins[NUMERICAS + CATEGORICAS]
y = pinguins[ALVO]
print('X:', X.shape, '| y:', y.shape)

# Parte A · A régua (cap. 5)
## A1 · Separar ANTES de preparar
Retiramos as 2 linhas sem nenhuma medida (não há o que imputar) e separamos 25% para teste **antes de qualquer preparação**. O `stratify=y` mantém a proporção das espécies nos dois lados. A partir daqui, o teste fica guardado.

In [ ]:
sem_medidas = pinguins[NUMERICAS].isna().all(axis=1)
X, y = X[~sem_medidas], y[~sem_medidas]
# TODO 1 (cap. 5): separe 25% para teste, estratificando por y e com random_state=SEMENTE
X_treino, X_teste, y_treino, y_teste = None, None, None, None
print('Treino:', X_treino.shape, '| Teste:', X_teste.shape)

## A2 · A régua mínima: baseline

In [ ]:
from sklearn.dummy import DummyClassifier
# TODO 2 (cap. 5): DummyClassifier que sempre chuta a classe mais frequente, ajustado no treino
baseline = None
print(f'Acurácia do baseline no teste: {baseline.score(X_teste, y_teste):.3f}')

**Leitura:** é o 44% da Unidade 1. Qualquer modelo daqui para frente precisa ficar **bem** acima disso para valer alguma coisa.

# Parte B · Preparar sem trapacear (cap. 6 e 7)
## B1 · Preparação dentro de um Pipeline (cap. 6)
Imputação, padronização e *one-hot* ficam dentro do `Pipeline`. Assim tudo é aprendido **só com o treino**, e o teste separado na Parte A continua protegido.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier

preparo = ColumnTransformer([
    ('num', Pipeline([('imputa', SimpleImputer(strategy='median')),
                      # TODO 3 (cap. 6): acrescente a etapa de padronização aqui
                      ]), NUMERICAS),
    ('cat', Pipeline([('imputa', SimpleImputer(strategy='most_frequent')),
                      ('codifica', OneHotEncoder(handle_unknown='ignore'))]), CATEGORICAS),
])
knn_com_escala = Pipeline([('preparo', preparo), ('modelo', KNeighborsClassifier(n_neighbors=5))])
knn_com_escala.fit(X_treino, y_treino)
print(f'k-NN com Pipeline completo: {knn_com_escala.score(X_teste, y_teste):.3f}')

## B2 · Por que a escala importa (cap. 7)
Mesmo modelo, mesmos dados, **uma** diferença: sem padronizar. Mudamos uma coisa por vez para saber o que causou a diferença.

In [ ]:
preparo_sem_escala = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), NUMERICAS),
    ('cat', Pipeline([('imputa', SimpleImputer(strategy='most_frequent')),
                      ('codifica', OneHotEncoder(handle_unknown='ignore'))]), CATEGORICAS),
])
knn_sem_escala = Pipeline([('preparo', preparo_sem_escala), ('modelo', KNeighborsClassifier(n_neighbors=5))])
knn_sem_escala.fit(X_treino, y_treino)

resultado = pd.DataFrame({
    'configuração': ['Baseline (classe mais frequente)', 'k-NN sem padronizar', 'k-NN com padronização'],
    'acurácia no teste': [baseline.score(X_teste, y_teste), knn_sem_escala.score(X_teste, y_teste),
                          knn_com_escala.score(X_teste, y_teste)]})
resultado.round(3)

In [ ]:
from sklearn.metrics import pairwise_distances
dois = X_treino[NUMERICAS].dropna().iloc[:2]
print('Dois pinguins:'); print(dois)
dif = (dois.iloc[0] - dois.iloc[1]) ** 2
print('\nContribuição de cada coluna para a distância² SEM padronizar (%):')
print((100 * dif / dif.sum()).round(1))

**Leitura:** a padronização levou o k-NN de cerca de 80% para 100% de acerto no teste. Acertar 100% aqui não indica vazamento: as três espécies são de fato bem separadas por essas medidas, como o pairplot sugeria. **Em dados reais de projeto, 100% é sinal de alerta** e pede investigação (cap. 6).

Sem padronizar, a massa corporal (em gramas) responde por quase toda a distância. O k-NN passa a escolher vizinhos quase só pelo peso, e as medidas do bico, que separavam bem as espécies, ficam praticamente ignoradas.

**Pergunta para discussão:** se ajustássemos o `StandardScaler` na base inteira antes do `train_test_split`, o que mudaria?

**Resposta esperada:** a média e o desvio seriam calculados também com os pinguins do teste. Seria um vazamento pequeno nestes dados, mas é um vício de método que, em outros problemas, infla bastante o resultado. Com o `Pipeline`, o problema não existe.

## B3 · Olhar os erros
A acurácia é um número só. A matriz de confusão mostra **quem** foi confundido com quem (a leitura completa das métricas vem na Parte D).

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(knn_com_escala, X_teste, y_teste, cmap='Blues')
plt.title('k-NN com padronização · teste'); plt.show()

# Parte C · Regressão (cap. 8)
Agora o y é uma **quantidade**: a progressão do diabetes um ano após a consulta. O protocolo é o mesmo da Parte A: separar, baseline, modelos dentro de Pipeline. A novidade é a **validação cruzada** (`KFold`): em vez de uma nota só, 5 notas no treino, sem tocar no teste.
## C1 · Dados e split

In [ ]:
from sklearn.datasets import load_diabetes
diab = load_diabetes(as_frame=True)
Xr, yr = diab.data, diab.target
print(Xr.shape); print(yr.describe().round(1))
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.2, random_state=SEMENTE)
cv_r = KFold(n_splits=5, shuffle=True, random_state=SEMENTE)

As 10 colunas (idade, sexo, IMC, pressão e 6 medidas do sangue) já vêm centradas e escaladas pelo próprio scikit-learn. Mesmo assim, mantemos o `StandardScaler` no pipeline por hábito: na base do seu projeto, isso não virá pronto.

## C2 · Baseline, Linear, Ridge e Lasso

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, LassoCV

def cv_regressao(modelo):
    r = cross_validate(modelo, Xr_tr, yr_tr, cv=cv_r,
                       scoring=['neg_mean_absolute_error', 'neg_root_mean_squared_error', 'r2'])
    return {'MAE': -r['test_neg_mean_absolute_error'].mean(),
            'RMSE': -r['test_neg_root_mean_squared_error'].mean(),
            'R2': r['test_r2'].mean()}

regressores = {
    # TODO 4 (cap. 5 e 8): o baseline da regressão, um DummyRegressor
    'Baseline (média)': None,
    'Linear': make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge (alpha=10)': make_pipeline(StandardScaler(), Ridge(alpha=10)),
    'Lasso (alpha por CV)': make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=SEMENTE)),
}
tabela_r = pd.DataFrame({n: cv_regressao(m) for n, m in regressores.items()}).T
tabela_r

**Leitura:** os três modelos lineares ficam muito parecidos e todos superam bem o baseline. O R² perto de 0,5 diz que metade da variação da doença é explicada pelas 10 medidas; a outra metade depende de coisas que não estão na base.

**Pergunta para discussão:** por que o RMSE é sempre maior que o MAE?

**Resposta esperada:** o RMSE eleva os erros ao quadrado antes de tirar a média, então erros grandes pesam mais. Se o RMSE for muito maior que o MAE, há alguns erros grandes; olhe os resíduos.

In [ ]:
lasso = regressores['Lasso (alpha por CV)'].fit(Xr_tr, yr_tr)
coefs = pd.Series(lasso[-1].coef_, index=Xr.columns)
print('alpha escolhido pelo LassoCV:', round(lasso[-1].alpha_, 3))
print('Coeficientes zerados pelo Lasso:', list(coefs[coefs == 0].index))
coefs.sort_values().plot.barh(figsize=(6, 3.5), title='Coeficientes do Lasso (dados padronizados)')
plt.axvline(0, color='grey', lw=0.8); plt.tight_layout(); plt.show()

**Regularização na prática (cap. 8):** o Lasso penaliza a soma dos valores absolutos dos coeficientes e, por isso, consegue **zerar** os menos úteis. É seleção de variáveis automática. Guarde a ideia: na Parte E, a teoria do cap. 10 explica por que restringir a complexidade do modelo ajuda a generalizar.

## C3 · O teste, uma vez, e os resíduos

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
final_r = regressores['Linear'].fit(Xr_tr, yr_tr)
pred = final_r.predict(Xr_te)
print(f'TESTE · Linear: MAE = {mean_absolute_error(yr_te, pred):.1f} | R2 = {r2_score(yr_te, pred):.3f}')
print(f'TESTE · Baseline: MAE = {mean_absolute_error(yr_te, np.full_like(pred, yr_tr.mean())):.1f}')
plt.scatter(pred, yr_te - pred, s=14); plt.axhline(0, color='grey')
plt.xlabel('Previsto'); plt.ylabel('Resíduo (real − previsto)'); plt.title('Resíduos no teste'); plt.show()

**Leitura dos resíduos:** se os pontos formam uma nuvem sem padrão em torno do zero, o modelo linear capturou o que podia. Um funil (resíduos crescendo com o previsto) ou uma curva indicariam que falta algo (transformar y, incluir interações ou usar um modelo não linear).

# Parte D · Classificação: logística e métricas (cap. 9)
## D1 · Dados e o que é "positivo"
No scikit-learn, `target = 0` é maligno. Recodificamos para que **1 = maligno**, a classe que não queremos deixar passar. As métricas de precisão e recall sempre se referem à classe positiva.

In [ ]:
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer(as_frame=True)
Xc = bc.data
yc = (bc.target == 0).astype(int)   # 1 = maligno
print(Xc.shape, '| proporção de malignos:', round(yc.mean(), 3))
# TODO 5 (cap. 5): split 75/25 ESTRATIFICADO por yc, random_state=SEMENTE
Xc_tr, Xc_te, yc_tr, yc_te = None, None, None, None
cv_c = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)

## D2 · Regressão logística e suas métricas

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xc_tr, yc_tr)
dummy = DummyClassifier(strategy='most_frequent').fit(Xc_tr, yc_tr)
print('Baseline (sempre benigno) · acurácia no treino-CV:',
      cross_validate(dummy, Xc_tr, yc_tr, cv=cv_c)['test_score'].mean().round(3))
pred_cv = cross_val_predict(logit, Xc_tr, yc_tr, cv=cv_c)
print(classification_report(yc_tr, pred_cv, target_names=['benigno', 'maligno'], digits=3))
ConfusionMatrixDisplay.from_predictions(yc_tr, pred_cv, display_labels=['benigno', 'maligno'], cmap='Blues')
plt.title('Logística · previsões fora da dobra (treino)'); plt.show()

**Leitura:** o baseline acerta cerca de 63% sem olhar nada, porque 63% dos casos são benignos. Ele tem recall zero para maligno, ou seja, é inútil. A logística erra poucos casos, e a matriz mostra **quais** erros: falsos negativos (maligno dito benigno) são os mais graves aqui.

## D3 · O limiar de decisão

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_auc_score, average_precision_score
proba_cv = cross_val_predict(logit, Xc_tr, yc_tr, cv=cv_c, method='predict_proba')[:, 1]
print('ROC-AUC:', round(roc_auc_score(yc_tr, proba_cv), 3), '| PR-AUC (average precision):',
      round(average_precision_score(yc_tr, proba_cv), 3))
linhas = []
for limiar in [0.5, 0.4, 0.3, 0.2, 0.1]:
    p = (proba_cv >= limiar).astype(int)
    vp = ((p == 1) & (yc_tr == 1)).sum(); fn = ((p == 0) & (yc_tr == 1)).sum(); fp = ((p == 1) & (yc_tr == 0)).sum()
    linhas.append({'limiar': limiar, 'recall': vp / (vp + fn), 'precisão': vp / (vp + fp),
                   'malignos perdidos (FN)': fn, 'alarmes falsos (FP)': fp})
pd.DataFrame(linhas)

**O limiar é uma decisão de negócio (cap. 9).** Baixar o limiar de 0,5 para 0,3 troca alguns alarmes falsos por menos malignos perdidos. Quem escolhe o limiar é o custo de cada erro no problema real, não o algoritmo.

**Pergunta para discussão:** num programa público de triagem, que limiar você defenderia, e o que precisa saber para decidir?

**Resposta esperada:** um limiar mais baixo, porque perder um maligno custa muito mais que um exame extra. Para decidir, é preciso saber a capacidade de exames confirmatórios (quantos alarmes o serviço aguenta) e o custo de cada tipo de erro.

# Parte E · Por que o treino engana (cap. 10)
Nas Partes C e D, o erro de treino foi sempre mais otimista que o da validação, e o Lasso "segurou" o modelo. A teoria do cap. 10 explica isso: quanto mais **capacidade** o modelo tem, mais ele consegue decorar o treino, e maior fica a distância para a validação. Uma árvore de decisão com profundidade crescente mostra isso num gráfico só (a árvore em si é assunto da Unidade 3).

In [ ]:
from sklearn.tree import DecisionTreeClassifier
prof = range(1, 13)
treino, valid = [], []
for d in prof:
    r = cross_validate(DecisionTreeClassifier(max_depth=d, random_state=SEMENTE), Xc_tr, yc_tr,
                       cv=cv_c, scoring='f1', return_train_score=True)
    treino.append(r['train_score'].mean()); valid.append(r['test_score'].mean())
plt.plot(prof, treino, marker='o', label='treino'); plt.plot(prof, valid, marker='o', label='validação (CV)')
plt.xlabel('max_depth'); plt.ylabel('F1 (maligno)'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Curva de complexidade da árvore'); plt.show()
print('Melhor profundidade na validação:', list(prof)[int(np.argmax(valid))])

**Leitura:** o F1 no treino chega a 1,0 (a árvore decora), mas a validação para de subir e oscila. A distância entre as curvas é o **sobreajuste**, e é por isso que só a validação cruzada pode escolher a profundidade. A regularização (o α do Lasso, o C da logística, a profundidade da árvore) é o jeito prático de controlar a capacidade.

**Pergunta para discussão:** por que não escolher a profundidade olhando a nota do **teste**?

**Resposta esperada:** porque o teste passaria a participar da escolha, e a nota final ficaria otimista. Quem escolhe hiperparâmetro é a validação cruzada no treino; o teste é aberto uma vez, no fim.

## Checklist da Unidade 2
- [ ] Separei o teste **antes** de preparar, estratificado quando é classificação.
- [ ] Tenho baseline e ele aparece na tabela.
- [ ] Toda preparação que usa estatística dos dados está dentro do `Pipeline`.
- [ ] Padronizei antes de qualquer método baseado em distância.
- [ ] Defini a classe positiva e escolhi a métrica pelo custo dos erros.
- [ ] Usei validação cruzada no treino e abri o teste uma única vez.
- [ ] Discuti o limiar de decisão.
- [ ] Sei explicar o sobreajuste pela curva de treino × validação.